# About Notebook 06_control_flow

## Tujuan Pembelajaran

* Membedakan tipe data mutable (bisa diubah) dan immutable (tidak bisa diubah), serta tahu konsekuensinya
* Memahami bahwa variabel di Python itu bukan kotak berisi nilai, melainkan label yang menunjuk ke objek di memori
* Mengenali kapan dua variabel berbagi referensi yang sama ke satu objek — dan kenapa itu bisa jadi sumber bug
* Membuat copy yang benar dari sebuah object, supaya tidak sengaja mengubah data asli
* Membedakan operator == (kesamaan nilai) dengan is (kesamaan identitas objek) — dan tahu kapan pakai yang mana
* Mendiagnosis dan memperbaiki bug klasik akibat shared reference

## Isi Materi

1. Mutable vs Immutable — konsep dasar, tipe mana yang termasuk kategori mana
2. Object Reference — variabel sebagai "label", bukan "kotak"
3. Copy — assignment vs .copy(), pengantar shallow copy
4. Equality vs Identity — == vs is, termasuk is None
5. Common Bugs — kumpulan jebakan klasik dari konsep di atas

## Goals

1. Menjelaskan kenapa mengubah list yang di-assign ke variabel lain, ikut mengubah variabel aslinya
2. Menjelaskan perbedaan == dan is dengan contoh sendiri, bukan hafalan definisi
3. Membuat copy list/dict yang benar-benar independen dari aslinya
4. Menemukan dan memperbaiki bug akibat shared reference pada kode sendiri
5. Selalu pakai is None (bukan == None) untuk mengecek nilai kosong

# 1. Mutable vs Immutable

## Konsep Mutability

* Mutable: isi object bisa diubah setelah dibuat, tanpa membuat object baru
* Immutable: isi object tidak bisa diubah sama sekali setelah dibuat

## Mutable

* list → bisa .append(), ubah elemen langsung, dll
* dict → bisa tambah/ubah/hapus key-value langsung
* set → bisa .add(), .remove() langsung

## Immuatble

* int, float → setiap "operasi" sebenarnya membuat angka baru, bukan mengubah yang lama
* str → sudah kamu pelajari di materi 02, setiap .replace() dll menghasilkan string baru
* tuple → sudah kamu pelajari di materi 04, sekali dibuat isinya final

In [1]:
# Immutable -> setiap "perubahan" sebenarnya membuat object BARU
angka = 5
print(id(angka))      # misal: 140712834957280 (alamat memori si objek 5)
angka = angka + 1
print(id(angka))       # ALAMAT BERBEDA! -> ini bukan '5' yang berubah jadi '6',
                         # tapi '6' adalah objek baru, dan 'angka' sekarang menunjuk ke situ

# Mutable -> perubahan terjadi PADA objek yang sama, tanpa buat objek baru
data = [1, 2, 3]
print(id(data))         # misal: 140712834958400
data.append(4)
print(id(data))          # ALAMAT SAMA! -> objeknya tetap sama, cuma isinya berubah

140707891004456
140707891004488
2168235291648
2168235291648


> id() adalah fungsi bawaan Python untuk melihat "alamat" objek di memori — akan sering dipakai di materi ini untuk membuktikan konsep secara konkret, bukan sekadar teori.

# 2. Object Reference

## Variable Reference

* Pada bahasa pemograman lain variabel dibayangkan sebagai kotak yang menyimpan nilai
* Di python variabel: lebel/nama yang menunjuk /reference ke sebuah objek yang tersimpan di memori
* Satu objek bisa ditunjuk oleh beberapa varibel

## Assignment

* Saat `a = [1, 2, 3]`, python membuat objek list memory, lalu label " a " ditempelkan ke object itu
* Saat `b = a`, python tidak membuat objek baru - b cuma menjadi label ke dua yang menunjuk ke object yang sama

## Shared reference

Karena `a` dan `b` menunjuk objek yang sama, mengubah lewat salah satu akan mengubah yang lain, karena satu objek

In [1]:
a = [1, 2, 3]
b = a

b.append(4)
print(a)

[1, 2, 3, 4]


In [2]:
print(id(a) == id(b))

True


![variabel_as_label_reference.png](../assets/variable_as_label_reference.png)

## Dampak perubahan object

Ini efek samping yang sering tidak disadari, terutama saat data dikirim ke fungsi

In [3]:
def tambah_item(daftar):
    daftar.append("item baru")
    return daftar

belanja = ['Susu', 'Beras']
tambah_item(belanja)
print(belanja)

['Susu', 'Beras', 'item baru']


> Kenapa penting: ini beda dari tipe immutable. Kalau kamu kirim int/str/tuple ke fungsi lalu "diubah" di dalamnya, variabel aslinya tidak ikut berubah — karena immutable selalu bikin objek baru. Tapi list/dict/set ikut berubah karena mutable diubah langsung di tempat (in-place).

# 3. Copy

## Assignment vs copy

* b = a → bukan copy, cuma menambah label baru ke objek yang sama (sudah dibahas di atas)
* Kalau kamu memang butuh objek baru yang independen, harus eksplisit pakai .copy()

## `.copy()`

In [4]:
a = [1, 2, 3]
b = a.copy()      # sekarang b adalah OBJEK BARU, isinya sama tapi terpisah

b.append(4)
print(a)   # [1, 2, 3] -> TIDAK berubah, karena b sudah objek terpisah
print(b)   # [1, 2, 3, 4]

print(id(a) == id(b))   # False -> dua objek berbeda

[1, 2, 3]
[1, 2, 3, 4]
False


Dictionary dan set juga punya .copy() dengan perilaku serupa

In [5]:
dict_asli = {"nama": "Andi", "umur": 20}
dict_copy = dict_asli.copy()

dict_copy["umur"] = 21
print(dict_asli)   # {'nama': 'Andi', 'umur': 20} -> tidak ikut berubah
print(dict_copy)   # {'nama': 'Andi', 'umur': 21}

{'nama': 'Andi', 'umur': 20}
{'nama': 'Andi', 'umur': 21}


## Shallow copy secara pengantar

* .copy() yang baru dibahas di atas disebut shallow copy (copy dangkal)
* Artinya: level terluar objek memang jadi baru dan independen, tapi kalau di  dalamnya ada objek mutable lagi (nested list/dict), level dalam itu masih berbagi referensi yang sama

## Awareness terhadap nested object

In [8]:
data_asli = [[1, 2], [3, 4]]
data_copy = data_asli.copy()      # shallow copy

data_copy.append([5, 6])           # ini aman, level luar independen
print(data_asli)   # [[1, 2], [3, 4]] -> tidak terpengaruh

data_copy[0].append(99)            # tapi ini MENGUBAH data_asli juga!
print(data_asli)   # [[1, 2, 99], [3, 4]] -> ikut berubah!
print(data_copy)   # [[1, 2, 99], [3, 4], [5, 6]]

[[1, 2], [3, 4]]
[[1, 2, 99], [3, 4]]
[[1, 2, 99], [3, 4], [5, 6]]


* Kenapa bisa begitu? Karena .copy() cuma menyalin "lapisan luar" — sub-list [1, 2] di dalamnya tetap objek yang sama, dipakai bersama oleh data_asli dan data_copy
* Solusi untuk nested object yang benar-benar independen di semua level (deep copy) akan dibahas lebih lengkap nanti pakai modul copy — untuk sekarang, cukup sadar bahwa .copy() biasa tidak selalu 100% aman untuk data bersarang

![Aperbedaan "sudah aman" vs "masih bug"](../assets/shared_reference_vs_copy.png)

# 4. Equality vs Identity

## `==`

* Mengecek apakah isi/nilai dua object sama
* Ini saat membandingkan data

In [9]:
a = [1, 2, 3]
b = [1, 2, 3]     # object BERBEDA, tapi isinya sama

print(a == b)   # True -> isinya sama persis

True


## `is`

* Mengecek apakah dua variabel menunjuk ke object yang benar-benar sama di memori (identitas, bukan isi)
* Ini yang berkaitan langsung dengan konsep reference yang baru kamu pelajari

In [10]:
print(a is b)          # False -> walau isinya sama, ini DUA OBJEK BERBEDA
print(id(a), id(b))    # alamat memori berbeda, membuktikan ini dua objek terpisah

c = a
print(a is c)   # True -> c cuma label baru untuk objek yang SAMA PERSIS dengan a

False
1783796240896 1783796241152
True


## Equality vs identity

| Operator | Mengecek                                                        | Kapan dipakai                                                                                   |
| -------- | --------------------------------------------------------------- | ----------------------------------------------------------------------------------------------- |
| `==`     | Kesamaan **nilai/isi**                                          | Membandingkan data (hampir selalu ini yang kamu mau)                                            |
| `is`     | Kesamaan **identitas objek** (alamat/referensi objek di memori) | Mengecek apakah dua variabel benar-benar merujuk ke objek yang sama, atau untuk mengecek `None` |


## is None

* Untuk mengecek apakah suatu variabel bernilai None, konvensi standar Python adalah pakai is None, bukan == None.
* Alasannya: None di Python hanya ada satu object tunggal di seluruh program (unik), jadi mengecek identitasnya (is) lebih tepat secara konsep dan sedikit lebih cepat.

In [2]:
data = None

# Cara yang BENAR (konvensi standar)
if data is None:
    print("Data kosong")

# Secara teknis == None juga jalan, tapi bukan gaya yang disarankan
if data == None:      # bekerja, tapi tidak Pythonic
    print("Data kosong")

Data kosong
Data kosong


![visual untuk membedakan == dan is](../assets/equality_vs_identity.png)

# 5. Common Bugs

## Shared reference (tidak sengaja)

In [3]:
# BUG: bermaksud bikin backup, tapi ternyata cuma nempel label
data_asli = [1, 2, 3]
backup = data_asli          # niatnya backup, tapi ini shared reference!

data_asli.append(4)
print(backup)   # [1, 2, 3, 4] -> "backup" ikut berubah, gagal jadi backup!

# PERBAIKAN
data_asli2 = [1, 2, 3]
backup2 = data_asli2.copy()   # sekarang benar-benar independen

data_asli2.append(4)
print(backup2)   # [1, 2, 3] -> backup aman

[1, 2, 3, 4]
[1, 2, 3]


## Mengubah object secara tidak sengaja (lewat fungsi)

In [ ]:
# BUG: fungsi mengubah list asli tanpa disadari si pemanggil fungsi

def proses_data(data):
    data.sort()          # ini mengubah list ASLI, bukan cuma dalam fungsi
    return data

nilai = [3, 1, 4, 1, 5]
hasil = proses_data(nilai)
print(nilai)   # [1, 1, 3, 4, 5] -> ikut berubah, walau tidak eksplisit ditulis begitu

# PERBAIKAN -> proses copy-nya, biar aslinya tidak tersentuh
def proses_data_aman(data):
    data_baru = data.copy()
    data_baru.sort()
    return data_baru

nilai2 = [3, 1, 4, 1, 5]
hasil2 = proses_data_aman(nilai2)
print(nilai2)   # [3, 1, 4, 1, 5] -> tidak berubah
print(hasil2)   # [1, 1, 3, 4, 5]

[1, 1, 3, 4, 5]
[3, 1, 4, 1, 5]
[1, 1, 3, 4, 5]


## Salah menggunakan is

In [ ]:
# BUG: pakai `is` untuk membandingkan NILAI, padahal maksudnya cek isi sama atau tidak


a = [1, 2, 3]
b = [1, 2, 3]

if a is b:                  # ini nyaris selalu False untuk list/dict beda object!
    print("Sama")
else:
    print("Beda")           # ini yang tercetak, walau isinya identik -> bug logika
    

Beda


Perbaikan > pakai `==` untuk membandingkan isi / nilai

In [6]:
if a == b:
    print("Sama")
else:
    print("Beda")

Sama


> Prinsip mengingat: pakai == untuk hampir semua kasus perbandingan data sehari-hari. Pakai is hanya untuk cek None, atau saat kamu memang secara spesifik ingin tahu apakah dua variabel adalah objek yang benar-benar sama persis di memori.

# Latihan

1. Eksperimen mutable list

    * Buat `list_a = [10, 20, 30]`
    * Buat `list_b = list_a` (bukan `.copy()`)
    * Cetak `id(list_a)` dan `id(list_b)`, buktikan apakah sama
    * Tambahkan elemen `40` ke `list_b` lewat `.append()`
    * Cetak `list_a` — jelaskan dengan kata-katamu sendiri kenapa hasilnya seperti itu

In [11]:
list_a = [10, 20, 30]
list_b = list_a

print(f"Isi list_a {list_a}")
print(f"Isi list_b {list_b}")

print(f"ID list_a {id(list_a)}")
print(f"ID list_b {id(list_b)}\n")

if list_a == list_b:
    print("ID Memory sama")
else:
    print("ID Memory Beda")
    
list_b.append(40)

print("\nCek list_a", list_a)

Isi list_a [10, 20, 30]
Isi list_b [10, 20, 30]
ID list_a 2900189937600
ID list_b 2900189937600

ID Memory sama

Cek list_a [10, 20, 30, 40]


walaupun nilai ditambahkan di var list_b, list_a ikut berubah karena id memory sama > list_b hanya label yang menempel di memori

2. Membandingkan object

    * Buat dua dictionary terpisah dengan isi persis sama: `dict_x = {"a": 1}` dan `dict_y = {"a": 1}`
    * Cetak hasil `dict_x == dict_y` dan `dict_x is dict_y`, jelaskan kenapa hasilnya berbeda
    * Buat `dict_z = dict_x`, cetak `dict_x is dict_z` — jelaskan bedanya dengan poin sebelumnya

In [12]:
dict_x = {"a": 1}
dict_y = {"a": 1}

In [13]:
print(dict_x == dict_y)
print(dict_x is dict_y)

True
False


In [14]:
dict_z = dict_x
print(dict_x is dict_z)

True


3. Membuat copy

    * Buat `data_asli = [1, 2, 3, 4, 5]`
    * Buat `data_copy` menggunakan `.copy()`
    * Ubah salah satu elemen di `data_copy` (misal `data_copy[0] = 99`)
    * Buktikan `data_asli` tidak berubah dengan mencetak keduanya

In [15]:
data_asli = [1, 2, 3, 4, 5]
data_copy = data_asli.copy()

In [17]:
data_copy[0] = 99
print(data_asli) 
print(data_copy)

[1, 2, 3, 4, 5]
[99, 2, 3, 4, 5]


4. Memperbaiki reference bug
    Diberikan kode bermasalah berikut:`

In [18]:
def tambah_bonus(nilai_siswa):
    nilai_siswa.append(10)  # bonus 10 poin
    return nilai_siswa

nilai_kelas_a = [70, 80, 90]
nilai_kelas_a_bonus = tambah_bonus(nilai_kelas_a)

print("Kelas A asli:", nilai_kelas_a)
print("Kelas A bonus:", nilai_kelas_a_bonus)

Kelas A asli: [70, 80, 90, 10]
Kelas A bonus: [70, 80, 90, 10]


* Jalankan dulu kode di atas, amati kenapa `nilai_kelas_a` (yang harusnya tidak disentuh) ikut berubah
* Perbaiki fungsi tambah_bonus supaya `nilai_kelas_a` (data asli) tidak ikut berubah, tapi `nilai_kelas_a_bonus` tetap dapat bonus 10 poin

In [19]:
def tambah_bonus(nilai_siswa):
    nilai_siswa_copy = nilai_siswa.copy()
    nilai_siswa_copy.append(10)  # bonus 10 poin
    return nilai_siswa_copy

nilai_kelas_a = [70, 80, 90]
nilai_kelas_a_bonus = tambah_bonus(nilai_kelas_a)

print("Kelas A asli:", nilai_kelas_a)
print("Kelas A bonus:", nilai_kelas_a_bonus)

Kelas A asli: [70, 80, 90]
Kelas A bonus: [70, 80, 90, 10]
